In [4]:
import numpy as np
import json
import hist
import matplotlib, matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from matplotlib.patches import FancyArrow
import matplotlib, matplotlib.pyplot as plt
matplotlib.use("pgf")
matplotlib.rcParams.update({
    "pgf.texsystem": "lualatex",
    "text.usetex": True,
    "pgf.rcfonts": False,  # Don't override with default matplotlib fonts
    "pgf.preamble": "", 
    #     "\\\\usepackage{fontspec}"
    #     "\\\\setmainfont{Computer Modern Roman}"  # or Computer Modern if installed
})
from matplotlib.lines import Line2D

textsize = 15

In [5]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Cosine-like PDFs
# ----------------------------
def pdf_sin_peak0(x):
    vals = np.cos(0.5 * np.pi * x)**2
    norm = np.trapz(vals, x)
    return vals / norm

def pdf_sin_peak1(x):
    vals = np.sin(0.5 * np.pi * x)**2
    norm = np.trapz(vals, x)
    return vals / norm

# ----------------------------
# Parameters
# ----------------------------
N = 50
x = np.linspace(0, 1, 500)

p = pdf_sin_peak0(x)  # peak at x=0
q = pdf_sin_peak1(x)  # peak at x=1

# ----------------------------
# Sampling
# ----------------------------
def sample_pdf(pdf_vals, x, n, rng=np.random.default_rng()):
    """Inverse transform sampling from discrete pdf_vals on grid x"""
    cdf = np.cumsum(pdf_vals)
    cdf = cdf / cdf[-1]
    u = rng.uniform(size=n)
    return np.interp(u, cdf, x)

rng = np.random.default_rng()
samples_p = sample_pdf(p, x, N, rng)
samples_q = sample_pdf(q, x, N, rng)

# ----------------------------
# Plot PDFs + samples
# ----------------------------
fig, ax1 = plt.subplots(figsize=(6,5))

# Left axis: PDFs
ax1.plot(x, p, linewidth=2, color="blue", label='$g(x)$')
ax1.plot(x, q, linewidth=2, color="red", label='$q(x)$')
ax1.set_xlabel("$x$", fontsize=15)
ax1.set_xlim(0,1)
ax1.set_ylabel("$\\mathrm{PDF}(x)$", fontsize=15)

max_y = max(p.max(), q.max())
ymin = -0.05 * max_y
ymax = 1.05 * max_y
ax1.set_ylim(ymin, ymax)

# Right axis: normalize LHS to [0,1]
ax2 = ax1.twinx()
ax2.set_ylim(ymin,ymax)
ax2.set_ylabel("$s(x)$", fontsize=15)
ax2.set_yticks([0,1])

ax1.tick_params(labelsize=12)
ax2.tick_params(labelsize=12)

# Sample markers
ax2.scatter(samples_p, np.zeros_like(samples_p),
            color="blue", alpha=0.25, edgecolor="k", s=40)
ax2.scatter(samples_q, np.ones_like(samples_q),
            color="red", alpha=0.25, edgecolor="k", s=40)

# Create dummy marker for arrow
legend_arrow = Line2D([0], [0], color='blue', marker='o', markersize=8,
                      linestyle='None', alpha=0.5)

# Collect handles and labels
handles, labels = ax1.get_legend_handles_labels()
handles.append(legend_arrow)
labels.append("$\\textrm{examples}$")
ax1.legend(handles=handles, labels=labels, frameon=False, fontsize=12)

plt.tight_layout()
plt.show()
plt.savefig('carl_diagram.pdf', bbox_inches='tight')


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

# ----------------------------
# Cosine-like PDF (normalized, goes to 0 at x=1)
# ----------------------------
def pdf_cos(x, offset=0.0):
    val = np.cos(0.5 * np.pi * (x + offset))
    vals = np.clip(val, 0, None)**2   # ensure nonnegative
    norm = np.trapz(vals, x)          # normalize over [0,1]
    return vals / norm

# ----------------------------
# Parameters
# ----------------------------
N_points = 10
x = np.linspace(0, 1, 200)

offset    = 0.0   # main
offset_u  = - 0.2  # up variation
offset_d  = + 0.2  # down variation

q_main = pdf_cos(x, offset)
q_up   = pdf_cos(x, offset_u)
q_down = pdf_cos(x, offset_d)

# Uniform x-points along the main curve
x_uniform = np.linspace(x[0], x[-1], N_points+2)[1:-1]
q_values  = np.interp(x_uniform, x, q_main)  # interpolate from q_main

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(6,5))

# Curves
ax.plot(x, q_main, linewidth=2, color="blue", label='$g(x | \\theta_0 )$', zorder=1)
ax.plot(x, q_up, linewidth=1., color="blue", alpha=0.5, linestyle='--', label='$ g( x | \\theta_1 )$', zorder=1)
ax.plot(x, q_down, linewidth=1., color="blue", alpha=0.5, linestyle=':', label='$ g ( x  | \\theta_2 )$', zorder=1)

y_max = max(q_main.max(), q_up.max(), q_down.max())
ax.set_ylim(0, 1.05 * y_max)
ax.set_xlim(0, 1.0)

ax.set_xlabel('$x$', fontsize=15)
ax.set_ylabel('$\\mathrm{PDF}(x)$', fontsize=15)

ax.tick_params(labelsize=12)

# Red dots along main curve
ax.scatter(x_uniform, q_values, color="blue", s=50, alpha=0.5, edgecolor="k",
           zorder=4)

# ----------------------------
# Precise arrows using FancyArrowPatch
# ----------------------------
arrowhead = dict(shrinkA=0., shrinkB=0., mutation_scale=10)

for i, (xi, yi) in enumerate(zip(x_uniform, q_values)):
    y_up   = np.interp(xi, x, q_up)
    y_down = np.interp(xi, x, q_down)

    # only the first arrow gets the legend label
    label_up   = "$\\mathrm{Auxiliary\\ information}$" if i == 0 else "_nolegend_"

    # dashed arrow to up variation
    a_up = FancyArrowPatch((xi, yi), (xi, y_up),
                           arrowstyle='-|>',
                           linewidth=1.,
                           linestyle='-',
                           color='blue',
                           alpha=0.25, 
                           zorder=3,
                           **arrowhead)
    ax.add_patch(a_up)

    # dotted arrow to down variation
    a_down = FancyArrowPatch((xi, yi), (xi, y_down),
                             arrowstyle='-|>',
                             linewidth=1.,
                             linestyle='-',
                             color='blue',
                             alpha=0.25, 
                             zorder=3,
                             **arrowhead)
    ax.add_patch(a_down)

# ----------------------------
# Add legend with a Unicode arrow for auxiliary info
# ----------------------------
# Create dummy marker for arrow
legend_arrow = Line2D([0], [0], color='blue', marker='>', markersize=8,
                      linestyle='None', alpha=0.25)

# legend_arrow = FancyArrow(0, 0, 0.5, 0, color='blue', alpha=0.25, width=0.02)

# Collect handles and labels
handles, labels = ax.get_legend_handles_labels()
handles.append(legend_arrow)
labels.append("$\\textrm{Aux. info.}$")

print(handles)
print(labels)

# Collect all handles/labels
# handles, labels = ax.get_legend_handles_labels()

# Reorder into 3-column layout
# Column 1: q
q_handles   = [handles[0]]
q_labels    = [labels[0]]

# Column 2: g(theta)
g_handles   = [handles[1], handles[2]]
g_labels    = [labels[1], labels[2]]

# Column 3: p(theta)
p_handles   = [handles[3], handles[4]]
p_labels    = [labels[3], labels[4]]

# Stack into order: q | g | p
new_handles = q_handles + g_handles + p_handles
new_labels  = q_labels  + g_labels  + p_labels

# Legend with 3 columns
ax.legend(new_handles, new_labels,
          frameon=False, fontsize=12, ncol=3)
#,


plt.tight_layout()
plt.show()
plt.savefig('taylr_diagram.pdf', bbox_inches='tight')


[<matplotlib.lines.Line2D object at 0x7fd8158bd2d0>, <matplotlib.lines.Line2D object at 0x7fd8158bf160>, <matplotlib.lines.Line2D object at 0x7fd8158bf460>, <matplotlib.lines.Line2D object at 0x7fd8166ef880>]
['$g(x | \\theta_0 )$', '$ g( x | \\theta_1 )$', '$ g ( x  | \\theta_2 )$', '$\\textrm{Aux. info.}$']


IndexError: list index out of range

In [15]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Carl + Taylor PDFs
# ----------------------------
def pdf_sin_peak0(x):
    vals = np.cos(0.5 * np.pi * x)**2
    norm = np.trapz(vals, x)
    return vals / norm  # defines q(x)

def pdf_cos(x, offset=0.0):
    vals = np.cos(0.5 * np.pi * (x + offset))**2
    norm = np.trapz(vals, x)
    return vals / norm  # defines g(x|θ)

def pdf_mixture(x, q_vals, g_vals, alpha=0.5):
    vals = alpha * q_vals + (1 - alpha) * g_vals
    norm = np.trapz(vals, x)
    return vals / norm

# ----------------------------
# Parameters
# ----------------------------
x = np.linspace(0, 1, 500)

offset_main = 1.0
offset_up   = 1.2
alpha       = 0.5  # mixture weight for q(x)

# PDFs
q_fixed   = pdf_sin_peak0(x)
g_main    = pdf_cos(x, offset_main)
g_up      = pdf_cos(x, offset_up)

# Mixtures (g + q)
p_main = pdf_mixture(x, q_fixed, g_main, alpha)
p_up   = pdf_mixture(x, q_fixed, g_up, alpha)

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(figsize=(5,5))

# q(x) component
ax.plot(x, q_fixed, linewidth=1., color="blue", alpha=0.25, zorder=1, label='$q(x)$')

# g(x|θ) components
ax.plot(x, g_up, linewidth=1, color="red", linestyle='--', alpha=0.25, label='$g(x| C)$', zorder=2)
ax.plot(x, g_main, linewidth=1, color="red", label='$g(x| 0)$', alpha=0.25, zorder=2)

# p(x|θ) mixtures
ax.plot(x, p_up, linewidth=2, color="green", linestyle='--', label='$p(x| C)$', zorder=3)
ax.plot(x, p_main, linewidth=2, color="green", label='$p(x| 0)$', zorder=3)

# Axes
ax.set_xlim(0, 1)
ax.set_ylim(0, 2.5)
ax.set_xlabel('$x$', fontsize=15)
ax.set_ylabel('$\\mathrm{PDF}(x)$', fontsize=15)
ax.tick_params(labelsize=12)

# Collect all handles/labels
handles, labels = ax.get_legend_handles_labels()

# Reorder into 3-column layout
# Column 1: q
q_handles   = [handles[0], Line2D([0], [0], color='black', marker='none', markersize=0,
                      linestyle='None', alpha=0.00)]
q_labels    = [labels[0], '']

# Column 2: g(theta)
g_handles   = [handles[1], handles[2]]
g_labels    = [labels[1], labels[2]]

# Column 3: p(theta)
p_handles   = [handles[3], handles[4]]
p_labels    = [labels[3], labels[4]]

# Stack into order: q | g | p
new_handles = q_handles + g_handles + p_handles
new_labels  = q_labels  + g_labels  + p_labels

# Legend with 3 columns
ax.legend(new_handles, new_labels,
          frameon=False, fontsize=12, ncol=3
          )


ax.set_yticks([0.0, 0.5, 1.0, 1.5, 2.0])
ax.set_xticks([0.0, 0.5, 1.0])

plt.tight_layout()
plt.savefig('mixture_diagram.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch

# ----------------------------
# Cosine-like PDFs
# ----------------------------
def pdf_sin_peak0(x):
    vals = np.cos(0.5 * np.pi * x)**2
    norm = np.trapz(vals, x)
    return vals / norm

def pdf_sin_peak1(x):
    vals = np.sin(0.5 * np.pi * x)**2
    norm = np.trapz(vals, x)
    return vals / norm

def pdf_cos(x, offset=0.0):
    vals = np.cos(0.5 * np.pi * (x + offset))**2
    norm = np.trapz(vals, x)
    return vals / norm

# ----------------------------
# Sampling function
# ----------------------------
def sample_pdf(pdf_vals, x, n, rng=np.random.default_rng()):
    cdf = np.cumsum(pdf_vals)
    cdf = cdf / cdf[-1]
    u = rng.uniform(size=n)
    return np.interp(u, cdf, x)

# ----------------------------
# Parameters
# ----------------------------
N_samples = 50
x = np.linspace(0,1,500)

# Carl diagram PDFs
q = pdf_sin_peak0(x)
g = pdf_sin_peak1(x)
rng = np.random.default_rng()
samples_q = sample_pdf(q, x, N_samples, rng)
samples_g = sample_pdf(g, x, N_samples, rng)

# Taylor diagram PDFs
offset_main = 1.0
offset_up   = 1.2
g_main = pdf_cos(x, offset_main)
g_up   = pdf_cos(x, offset_up)
x_uniform = np.linspace(x[0], x[-1], 10+2)[1:-1]
g_values = np.interp(x_uniform, x, g_main)

# ----------------------------
# Create figure with (2,1) subplots, shared x
# ----------------------------
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5,8), sharex=True)

# ============================
# Top subplot: Carl diagram
# ============================
ax1.plot(x, q, linewidth=2, color="blue", label='$q(x)$')
ax1.plot(x, g, linewidth=2, color="red", label='$g(x | 0)$')
ax1.set_ylabel("$\\mathrm{PDF}(x)$", fontsize=15)

# Right axis for samples
ax1_r = ax1.twinx()
ax1_r.set_ylim(-0.05, 1.05)
ax1_r.set_yticks([0,1])
ax1_r.set_ylabel("$s(x)$", fontsize=15)
ax1_r.scatter(samples_q, np.ones_like(samples_q),
              color="blue", alpha=0.25, edgecolor="blue", s=40)
ax1_r.scatter(samples_g, np.zeros_like(samples_g),
              color="red", alpha=0.25, edgecolor="red", s=40)

# Dummy marker for legend
legend_arrow = Line2D([0], [0], color='red', marker='o', markersize=8,
                      linestyle='None', alpha=0.25)

handles, labels = ax1.get_legend_handles_labels()
handles.append(legend_arrow)
labels.append("$\\textrm{examples}$")
ax1.legend(handles=handles, labels=labels, frameon=False, fontsize=12)
ax1.tick_params(labelsize=12)
# ax1.set_yticks([0,0.5,1.0,1.5,2.0])

# ============================
# Bottom subplot: Taylor diagram
# ============================
ax2.plot(x, g_up, linewidth=1, color="red", alpha=0.5, linestyle='--', label='$g(x| C)$')
ax2.plot(x, g_main, linewidth=2, color="red", label='$g(x| 0)$')
ax2.scatter(x_uniform, g_values, color="red", s=50, alpha=0.5, edgecolor="red")

arrowhead = dict(shrinkA=0., shrinkB=0., mutation_scale=10)
for i, (xi, yi) in enumerate(zip(x_uniform, g_values)):
    y_up = np.interp(xi, x, g_up)
    a_up = FancyArrowPatch((xi, yi), (xi, y_up),
                           arrowstyle='-|>', linewidth=1,
                           color='red', alpha=0.25, zorder=3, **arrowhead)
    ax2.add_patch(a_up)

# Dummy arrow for legend
legend_arrow = Line2D([0], [0], color='red', marker='>', markersize=8,
                      linestyle='None', alpha=0.25)
handles, labels = ax2.get_legend_handles_labels()
handles.append(legend_arrow)
labels.append("$\\textrm{auxiliary information}$")
ax2.legend(handles=handles, labels=labels, frameon=False, fontsize=12)
ax2.set_xlabel("$x$", fontsize=15)
ax2.set_ylabel("$\\mathrm{PDF}(x)$", fontsize=15)
ax2.tick_params(labelsize=12)

ax2.tick_params(top=True, labeltop=False)
ax2.set_xlim(0,1)

ax1.set_yticks([0.0, 0.5, 1.0, 1.5, 2.0])
ax2.set_yticks([0.0, 0.5, 1.0, 1.5, 2.0])

ax1.set_xticks([0.0, 0.5, 1.0])

fig.tight_layout()
fig.subplots_adjust(hspace=0)
plt.savefig('carl_taylr_diagram.pdf', bbox_inches='tight')
